# Nemotron Reasoning — Baseline SFT Notebook w/ Unsloth

Unsloth is an optimization library that makes finetuning faster and uses lower VRAM. Using Unsloth, we can actually train on the whole dataset within a reasonable time (~1hr/epoch).

Fun fact: Unsloth has an actual guide for Nemotron 3 models - https://unsloth.ai/docs/models/nemotron-3

For training with this notebook, you will need a bunch of libraries other than Unsloth, the most important of which is mamba-ssm

## 1. Setup

In [1]:
!pip install -q --no-index --find-links /kaggle/input/datasets/mayukh18/nemotron-packages/packages unsloth trl peft transformers datasets accelerate bitsandbytes 
!pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
!pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


In [2]:
import os
import re
import math
from pathlib import Path
import zipfile
import pandas as pd
from datasets import Dataset
import kagglehub
import torch
from torch import nn
import bitsandbytes as bnb
from unsloth import FastLanguageModel
from sklearn.model_selection import train_test_split

DATA_DIR = Path("/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH  = DATA_DIR / "test.csv"
ADAPTER_DIR = "nemotron-lora-adapter"
SUBMISSION_ZIP = "submission.zip"

FRESH_START = True

if FRESH_START:
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
else:
    MODEL_PATH = "/kaggle/input/datasets/mayukh18/nemotron-lora-submission/submission/nemotron-lora-adapter"

# LoRA config
LORA_RANK    = 32      # competition allows up to 32
LORA_ALPHA   = 16
LORA_DROPOUT = 0.05

# Training config
MAX_SEQ_LEN = 1024 # was 768
NUM_EPOCHS  = 1
BATCH_SIZE  = 6
GRAD_ACCUM  = 4
LR          = 2e-4

MAX_NEW_TOKENS = 512
TEMPERATURE    = 0.0

print("Config ready.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Config ready.


## 2. Load & Inspect Data

In [3]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

train_df, val_df = train_test_split(train_df, test_size=100, random_state=42)

print(f"Train: {len(train_df):,} rows — columns: {list(train_df.columns)}")
print(f"Test:  {len(val_df):,} rows  — columns: {list(val_df.columns)}")
print(f"Test:  {len(test_df):,} rows  — columns: {list(test_df.columns)}")
train_df.head()

Train: 9,400 rows — columns: ['id', 'prompt', 'answer']
Test:  100 rows  — columns: ['id', 'prompt', 'answer']
Test:  3 rows  — columns: ['id', 'prompt']


,id,prompt,answer
6521,af0e4262,"In Alice's Wonderland, a secret set of transfo...",]{
7809,d14923e2,"In Alice's Wonderland, the gravitational const...",139.69
6478,addd2e88,"In Alice's Wonderland, a secret unit conversio...",15.48
7461,c81411a2,"In Alice's Wonderland, a secret set of transfo...",05
3971,6a9cf9e1,"In Alice's Wonderland, secret encryption rules...",dragon found the secret mirror


In [4]:
# Quick look at prompt length distribution
train_df["prompt_len"] = train_df["prompt"].str.len()
print(train_df["prompt_len"].describe())

# Sample one puzzle
sample = train_df.sample(1).iloc[0]
print("\n--- Sample Prompt ---")
print(sample["prompt"])
print("\n--- Answer ---")
print(sample["answer"])

count    9400.000000
mean      301.305106
std       104.115002
min       177.000000
25%       209.000000
50%       281.000000
75%       370.250000
max       510.000000
Name: prompt_len, dtype: float64

--- Sample Prompt ---
In Alice's Wonderland, secret encryption rules are used on text. Here are some examples:
adixwp cinjvmf nvm ydkxvn awwd -> dragon watches the bright door
uwqfm fnqakmf nvm ipjkmpn fmjdmn -> mouse studies the ancient secret
nvm uzfnmdkwqf diyykn dmiaf -> the mysterious rabbit reads
uwqfm adicf fmjdmn -> mouse draws secret
Now, decrypt the following text: opkxvn jdminmf qpamd rieijm

--- Answer ---
knight creates under palace


## 3. Format Data

We mirror the exact prompt format used in the competition's evaluation metric so there is **no train/inference mismatch**.

The metric notebook appends this instruction to every user message:
```
\nPlease put your final answer inside `\boxed{}`. For example: `\boxed{your answer}`
```

For training, the assistant response is simply `\boxed{answer}`.  
A chain-of-thought prefix can be added in a later iteration.

In [5]:
BOXED_INSTRUCTION = (
    "\nPlease put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)

def format_train_row(prompt: str, answer: str) -> dict:
    """Return a messages list for SFT chat-template formatting."""
    return {
        "messages": [
            {"role": "user",      "content": prompt + BOXED_INSTRUCTION},
            {"role": "assistant", "content": f"\\boxed{{{answer}}}"},
        ]
    }

def format_test_row(prompt: str) -> dict:
    """Return a messages list for inference (no answer)."""
    return {
        "messages": [
            {"role": "user", "content": prompt + BOXED_INSTRUCTION},
        ]
    }

# Build HuggingFace Dataset for training
train_records = [
    format_train_row(row["prompt"], str(row["answer"]))
    for _, row in train_df.iterrows()
]
train_dataset = Dataset.from_list(train_records)

# Build HuggingFace Dataset for validation
val_records = [
    format_train_row(row["prompt"], str(row["answer"]))
    for _, row in val_df.iterrows()
]
val_dataset = Dataset.from_list(val_records)

print(f"Formatted {len(train_dataset):,} train examples, {len(val_dataset):,} val examples.")
print("Sample messages:\n", train_dataset[0]["messages"])

Formatted 9,400 train examples, 100 val examples.
Sample messages:
 [{'content': 'In Alice\'s Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:\n>{*{\\ = \\/>$\n//*@$ = {@>$\n$\'*/\\ = {@"$\n){*/$ = "">]\n>>*>{ = /\\@]\nNow, determine the result for: "@-/$\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`', 'role': 'user'}, {'content': '\\boxed{]{}', 'role': 'assistant'}]


## 4. Load Base Model with LoRA (4-bit via Unsloth)

In [6]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = MAX_SEQ_LEN, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    trust_remote_code = True,
    unsloth_force_compile = False,
    attn_implementation = "eager",
    torch_dtype=torch.bfloat16,
    dtype=None,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.10: Fast Nemotron_H patching. Transformers: 5.3.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1 does not have a padding token! Will use pad_token = <SPECIAL_999>.


In [7]:
# del model, tokenizer

# import gc
# gc.collect()
# torch.cuda.empty_cache()

In [8]:
target_modules = [
    'out_proj', 'v_proj', 'q_proj', 'down_proj', 'embed_tokens',
    'k_proj', 'in_proj', 'up_proj', 'o_proj', 'lm_head', 'gate_proj'
]

In [9]:
if FRESH_START:
    # Apply LoRA adapter
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=target_modules,
        bias="none",
        use_gradient_checkpointing="unsloth",  # saves VRAM
        random_state=42,
    )

model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Unsloth: Detected MoE model with num_experts = 128 and target_modules = ['out_proj', 'v_proj', 'q_proj', 'down_proj', 'embed_tokens', 'k_proj', 'in_proj', 'up_proj', 'o_proj', 'lm_head', 'gate_proj']. Enabling LoRA on MoE parameters: ['mlp.experts.gate_up_proj', 'mlp.experts.down_proj']
Unsloth: PEFT set target_parameters but found no matching parameters.
This is expected for MoE models - Unsloth handles MoE expert LoRA targeting separately.
Unsloth: Making `model.base_model.model.backbone` require gradients
trainable params: 888,154,112 || all params: 32,466,091,456 || trainable%: 2.7356


## 5. Train with SFTTrainer

In [10]:
from trl import SFTTrainer
from transformers import TrainingArguments

train_dataset = train_dataset.map(lambda ex: {
    "text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False
    )
})

val_dataset = val_dataset.map(lambda ex: {
    "text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False
    )
})

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=TrainingArguments(
        output_dir="./nemotron-lora-checkpoints",
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1, #0.03,
        bf16=True,               # use fp16=True if bf16 unavailable
        logging_steps=50,
        save_strategy="epoch",
        optim="adamw_torch",  #"adamw_8bit",      # 8-bit Adam from bitsandbytes
        seed=42,
        report_to="none",        # disable wandb/tensorboard for baseline
        dataloader_num_workers=4,
        eval_strategy="steps",
        eval_steps=50,
    ),
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field=None,      # use messages format
    dataset_kwargs={"skip_prepare_dataset": False},
)

print("Starting training...")
trainer.train()
print("Training complete.")

Map:   0%|          | 0/9400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/9400 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/100 [00:00<?, ? examples/s]

Starting training...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 11}.


Step,Training Loss,Validation Loss
50,6.969706,0.756285
100,3.152039,0.708772
150,2.995034,0.688643
200,3.021647,0.674788
250,2.819132,0.665439
300,2.755900,0.662341
350,2.842409,0.658350


Unsloth: Not an error, but NemotronHForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


Training complete.


## 6. Save LoRA Adapter

In [11]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Verify adapter_config.json is present (required by competition)
assert os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")), \
    "adapter_config.json missing!"

print(f"Adapter saved to ./{ADAPTER_DIR}/")
print("Files:", os.listdir(ADAPTER_DIR))

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


Adapter saved to ./nemotron-lora-adapter/
Files: ['adapter_model.safetensors', 'adapter_config.json', 'tokenizer.json', 'chat_template.jinja', 'README.md', 'tokenizer_config.json']


## 7. Local Validation

Run inference on a small held-out slice of the training set to get a quick local accuracy estimate before submitting.

In [12]:
def extract_final_answer(text: str | None) -> str:
    r"""Extract the final answer from model output, prioritising \boxed{}."""
    if text is None:
        return "NOT_FOUND"

    # Prefer \boxed{...}
    matches = re.findall(r"\\boxed\{([^}]*)(?:\}|$)", text)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        return non_empty[-1] if non_empty else matches[-1].strip()

    # Common fallback patterns
    patterns = [
        r"The final answer is:\s*([^\n]+)",
        r"Final answer is:\s*([^\n]+)",
        r"Final answer\s*[:：]\s*([^\n]+)",
    ]
    for pattern in patterns:
        m = re.findall(pattern, text, re.IGNORECASE)
        if m:
            return m[-1].strip()

    # Last numeric value
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    if nums:
        return nums[-1]

    # Last non-empty line
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return lines[-1] if lines else "NOT_FOUND"


def verify(stored_answer: str, predicted: str) -> bool:
    """Return True if predicted matches stored_answer (numeric or string)."""
    stored_answer = stored_answer.strip()
    predicted = predicted.strip()
    try:
        return math.isclose(float(stored_answer), float(predicted),
                            rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        return predicted.lower() == stored_answer.lower()


print("Helper functions ready.")

Helper functions ready.


In [13]:
# # Switch model to inference mode (Unsloth optimisation)
# FastLanguageModel.for_inference(model)

# # Sample 100 examples from train for local eval
# eval_df = train_df.sample(20, random_state=42).reset_index(drop=True)

# correct = 0
# for _, row in eval_df.iterrows():
#     user_content = row["prompt"] + BOXED_INSTRUCTION
#     # Apply the tokenizer's chat template (system prompt optional for baseline)
#     messages = [{"role": "user", "content": user_content}]

#     text = tokenizer.apply_chat_template(
#         messages,
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     input_ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)

#     output_ids = model.generate(
#         input_ids=input_ids,
#         max_new_tokens=MAX_NEW_TOKENS,
#         temperature=TEMPERATURE if TEMPERATURE > 0 else None,
#         do_sample=TEMPERATURE > 0,
#         pad_token_id=tokenizer.eos_token_id,
#     )
#     # Decode only newly generated tokens
#     generated = tokenizer.decode(
#         output_ids[0][input_ids.shape[1]:],
#         skip_special_tokens=True
#     )

#     pred = extract_final_answer(generated)
#     if verify(str(row["answer"]), pred):
#         correct += 1

# local_acc = correct / len(eval_df)
# print(f"Local accuracy on {len(eval_df)} train samples: {local_acc:.2%}")

## 8. Submission

The competition expects a zip archive containing the LoRA adapter directory (with `adapter_config.json` at the root of the archive or a sub-directory).

In [14]:
!rm -rf /kaggle/working/nemotron-lora-checkpoints

In [15]:
with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(ADAPTER_DIR):
        zf.write(
            os.path.join(ADAPTER_DIR, fname),
            arcname=os.path.join(ADAPTER_DIR, fname),
        )

# Verify
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zf:
    names = zf.namelist()

has_config = any("adapter_config.json" in n for n in names)
print(f"Files in {SUBMISSION_ZIP}: {names}")
print(f"adapter_config.json present: {has_config}")

Files in submission.zip: ['nemotron-lora-adapter/adapter_model.safetensors', 'nemotron-lora-adapter/adapter_config.json', 'nemotron-lora-adapter/tokenizer.json', 'nemotron-lora-adapter/chat_template.jinja', 'nemotron-lora-adapter/README.md', 'nemotron-lora-adapter/tokenizer_config.json']
adapter_config.json present: True
